# PrivacyAlert — Data Preparation

Ce notebook constitue la **première étape du pipeline** : chargement, nettoyage et fusion de toutes les sources de features disponibles.

Le `df` final produit en sortie contient, pour chaque image :
- son `image_id` et son label de privacy (`public` / `private`)
- les informations de split (Fold 0 à Fold 9)
- les features de tags (user + deep tags encodés en one-hot)
- les features de scènes (scores de probabilité par type de scène)
- les features d'objets (score de confiance max par catégorie d'objet)

---

**Sources de données :**
```
data/
├── curated_imageprivacy_datasets/curated_privacyalert/
│   ├── annotations/labels_splits.csv
│   └── annotations/tags/user_deep_tags/dt_plus_ut_overall1_2.tsv
├── objects_privacyalert/PrivacyAlert/dets/
└── scenes_privacyalert/PrivacyAlert/scenes/
```

## Imports

In [2]:
import pandas as pd
import numpy as np
import glob
import json
import re
from sklearn.preprocessing import MultiLabelBinarizer

## Paths

Centralisation des chemins vers les données. Modifier ici si la structure de dossiers change.

In [1]:
tags_path            = "data/curated_imageprivacy_datasets/curated_privacyalert/annotations/tags/"
splits_path          = "data/curated_imageprivacy_datasets/curated_privacyalert/"
object_feature_path  = "data/objects_privacyalert/PrivacyAlert/dets/"
scenes_features_path = "data/scenes_privacyalert/PrivacyAlert/scenes/"

## 1. Labels

Chargement de `labels_splits.csv` qui contient à la fois :
- le label de privacy (`Label 2-class`, déjà encodé en entiers : 0 = public, 1 = private, -1 = non annotée)
- les affectations de folds pour la cross-validation (colonnes `Fold 0` à `Fold 9`)

On renomme les colonnes pour harmoniser avec le reste du pipeline :
- `Image Name` → `image_id`
- `Label 2-class` → `label`

Les 7 images non annotées (`label == -1`) sont retirées.

In [3]:
labels = pd.read_csv(
    splits_path + "annotations/labels_splits.csv"
)

labels = labels.rename(columns={
    "Image Name": "image_id",
    "Label 2-class": "label"
})

labels["image_id"] = labels["image_id"].astype(str)

# La colonne `label` est déjà encodée en entiers : 0 = public, 1 = private,
# -1 = non annotée. On retire les images non annotées.
n_before = len(labels)
labels = labels[labels["label"].isin([0, 1])].reset_index(drop=True)
labels["label"] = labels["label"].astype(int)

print(f"Labels chargés : {len(labels)} images ({n_before - len(labels)} non annotées retirées)")
print(f"Distribution : {labels['label'].value_counts().to_dict()}")
print(f"Colonnes : {list(labels.columns)}")

Labels chargés : 6793 images (7 non annotées retirées)
Distribution : {0: 5090, 1: 1703}
Colonnes : ['ID', 'image_id', 'batch', 'label', 'Final', 'Fold 0', 'Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5', 'Fold 6', 'Fold 7', 'Fold 8', 'Fold 9', 'Original']


## 2. Tags (user + deep)

On charge `dt_plus_ut_overall1_2.tsv` qui contient la combinaison des **user tags** (métadonnées Flickr) et des **deep tags** (tags visuels automatiques produits par des modèles pré-entraînés sur ImageNet et Places365).

Pipeline de traitement :
1. Renommage des colonnes (le fichier TSV n'a pas d'en-tête explicite)
2. Nettoyage des tokens : passage en minuscule, suppression des caractères non-alphabétiques, filtrage des tokens de longueur ≤ 2
3. Encodage one-hot via `MultiLabelBinarizer`
4. Filtrage des tags rares : on ne conserve que les tags présents dans **au moins 10 images**

In [4]:
user_deep_tags_overall = pd.read_csv(
    tags_path + "user_deep_tags/dt_plus_ut_overall1_2.tsv",
    sep="\t"
)

print(f"Tags chargés : {len(user_deep_tags_overall)} lignes")

Tags chargés : 6799 lignes


In [5]:
tags = user_deep_tags_overall.copy()
tags.columns = ["idx", "label_tmp", "image_id", "tags_text"]
tags = tags[["image_id", "tags_text"]]
tags["image_id"] = tags["image_id"].astype(str)

# Tokenisation et nettoyage
def clean(lst):
    return [re.sub(r"[^a-zA-Z]", "", w.lower()) for w in lst if len(w) > 2]

tags["tags_list"] = tags["tags_text"].apply(lambda x: x.split())
tags["tags_list"] = tags["tags_list"].apply(clean)

# Encodage one-hot
mlb = MultiLabelBinarizer()
tags_encoded = pd.DataFrame(
    mlb.fit_transform(tags["tags_list"]),
    columns=mlb.classes_,
    index=tags.index
)
tags_encoded["image_id"] = tags["image_id"].values

# Filtrage des tags rares (< 10 occurrences)
tag_counts = tags_encoded.drop(columns="image_id").sum()
valid_tags = tag_counts[tag_counts >= 10].index
tags_encoded = tags_encoded[["image_id"] + list(valid_tags)]

print(f"Tags encodés : {len(tags_encoded)} images, {len(valid_tags)} tags valides (>= 10 occurrences)")

Tags encodés : 6799 images, 3724 tags valides (>= 10 occurrences)


## 3. Scènes

Les features de scènes sont des **probabilités prédites** par un modèle CNN entraîné sur Places365. Chaque image est représentée par un vecteur de scores indiquant la probabilité d'appartenir à chaque type d'environnement (`bedroom`, `office`, `street`, etc.).

Chaque image dispose d'un fichier CSV individuel. On charge tous les fichiers, on les concatène, puis on pivote pour obtenir une ligne par image avec une colonne par type de scène.

In [6]:
scene_files = glob.glob(scenes_features_path + "**/*.csv", recursive=True)

dfs = []
for f in scene_files:
    df_tmp = pd.read_csv(f, sep=";")
    image_id = f.split("/")[-1].replace(".csv", "")
    df_tmp["image_id"] = image_id
    dfs.append(df_tmp)

scenes_long = pd.concat(dfs, ignore_index=True)

# Pivot : une ligne par image, une colonne par scène
scenes = scenes_long.pivot_table(
    index="image_id",
    columns="scene",
    values="conf"
).fillna(0).reset_index()

scenes["image_id"] = scenes["image_id"].astype(str)
scenes.columns.name = None  # nettoyage du nom d'index de colonnes issu du pivot

print(f"Scènes chargées : {len(scenes)} images, {scenes.shape[1] - 1} types de scènes")

Scènes chargées : 6800 images, 365 types de scènes


## 4. Objets

Les features d'objets proviennent de détections au format JSON (type COCO), produites par un CNN entraîné sur ImageNet. Chaque fichier JSON correspond à une image et contient les catégories d'objets détectés avec leur score de confiance.

Traitement : pour chaque image, on agrège les détections en conservant le **score de confiance maximum** par catégorie d'objet (une même catégorie peut apparaître plusieurs fois dans une image).

In [7]:
object_files = glob.glob(object_feature_path + "**/*.json", recursive=True)

rows = []
for f in object_files:
    with open(f) as file:
        data = json.load(file)

    image_id = f.split("/")[-1].replace(".json", "")

    # Score max par catégorie d'objet
    obj_dict = {}
    for cat, conf in zip(data["categories"], data["confidence"]):
        obj_dict[cat] = max(obj_dict.get(cat, 0), conf)

    obj_dict["image_id"] = image_id
    rows.append(obj_dict)

objects = pd.DataFrame(rows).fillna(0)
objects["image_id"] = objects["image_id"].astype(str)

print(f"Objets chargés : {len(objects)} images, {objects.shape[1] - 1} catégories d'objets")

Objets chargés : 6800 images, 79 catégories d'objets


## 5. Fusion

On part de `labels` comme table de référence et on y joint les trois familles de features via l'`image_id`.

Les jointures sont faites en `left` pour conserver toutes les images du dataset, même si certaines features sont manquantes. Les valeurs manquantes sont remplacées par 0 (absence de tag / objet / scène).

In [8]:
df = labels.copy()

df = df.merge(tags_encoded, on="image_id", how="left")
df = df.merge(scenes,      on="image_id", how="left")
df = df.merge(objects,     on="image_id", how="left")

df = df.fillna(0)

print(f"DataFrame final : {df.shape[0]} images, {df.shape[1]} colonnes")
print(f"Distribution labels : {df['label'].value_counts().to_dict()}")

DataFrame final : 6793 images, 4184 colonnes
Distribution labels : {0: 5090, 1: 1703}


## 6. Vérification rapide

Aperçu du DataFrame final et vérification de la cohérence des données avant export.

In [9]:
# Colonnes disponibles par famille

meta_cols  = ["image_id", "label"] + [c for c in df.columns if str(c).startswith("Fold")]
tag_cols   = [c for c in df.columns if c in set(valid_tags)]
scene_cols = [c for c in df.columns if c in set(scenes.columns) and c != "image_id"]
obj_cols   = [c for c in df.columns if c in set(objects.columns) and c != "image_id"]

print(f"Colonnes meta      : {len(meta_cols)}")
print(f"Colonnes tags      : {len(tag_cols)}")
print(f"Colonnes scènes    : {len(scene_cols)}")
print(f"Colonnes objets    : {len(obj_cols)}")
print()
print("Aperçu :")
df[meta_cols].head()

Colonnes meta      : 12
Colonnes tags      : 3668
Colonnes scènes    : 309
Colonnes objets    : 79

Aperçu :


,image_id,label,Fold 0,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Fold 6,Fold 7,Fold 8,Fold 9
0,51027421626,0,0,0,0,0,1,0,0,0,0,0
1,50668571902,0,0,0,0,1,0,0,0,0,0,0
2,51189940756,1,0,0,0,0,0,0,0,0,0,1
3,51196330717,0,2,2,2,2,2,2,2,2,2,2
4,51192228348,0,0,0,0,0,0,0,0,0,0,1


## 7. Export

Export du DataFrame final en **CSV**.

Le fichier exporté peut être directement utilisé par les notebooks de modélisation sans relancer ce pipeline.

In [ ]:
df.to_csv("data/privacyalert_features.csv", index=False)

print("Export terminé :")
print("  → data/privacyalert_features.csv")